In [0]:
df = spark.read.table("uber.bronze.rides_raw")
df.display()

In [0]:
df = spark.sql("select * from uber.bronze.bulk_rides")
df.schema


In [0]:
rides_Schema =df.schema

In [0]:
from pyspark.sql.functions import *

df = spark.sql("select * from uber.bronze.rides_raw")
df_parsed = df.withColumn("parsed_rides",from_json(col("rides"),rides_Schema)).select("parsed_rides.*")
df_parsed.display()


In [0]:
%sql
select * from uber.bronze.stg_rides;

In [0]:
pip install jinja2


In [0]:
jinja_config =[
    {
        "table" : "uber.bronze.stg_rides as stg_rides",
        "select" : "stg_rides.*",
        "where" : ""
    },
    {
        "table" : "uber.bronze.map_vehicle_makes as map_vehicle_makes",
        "select" : "map_vehicle_makes.*",
        "where" : "",
        "on": "stg_rides.vehicle_make_id = map_vehicle_makes.vehicle_make_id"
    },
    {
        "table" : "uber.bronze.map_vehicle_types as map_vehicle_types",
        "select" : "map_vehicle_types.vehicle_type,description,base_rate,per_mile,per_minute",
        "where" : "",
        "on": "stg_rides.vehicle_type_id = map_vehicle_types.vehicle_type_id"
        
    },
    {
        "table" : "uber.bronze.map_payment_methods as map_payment_methods",
        "select" : "map_payment_methods.*",
        "where" : "",
        "on": "stg_rides.payment_method_id = map_payment_methods.payment_method_id"
    },
    {
        "table" : "uber.bronze.map_ride_statuses as map_ride_statuses",
        "select" : "map_ride_statuses.*",
        "where" : "",
        "on": "stg_rides.ride_status_id = map_ride_statuses.ride_status_id"
    },
    {
        "table" : "uber.bronze.map_cancellation_reasons as map_cancellation_reasons",
        "select" : "map_cancellation_reasons.*",
        "where" : "",
        "on": "stg_rides.cancellation_reason_id = map_cancellation_reasons.cancellation_reason_id"

    },
    {
        "table" : "uber.bronze.map_cities as map_cities",
        "select" : "map_cities.*",
        "where" : "",
        "on": "stg_rides.pickup_city_id = map_cities.city_id"
    },
    {
        "table": "uber.bronze.map_rides as map_rides",
        "select" : "map_rides.*", 
        "where" : "",
        "on": "stg_rides.cancellation_reason_id = map_rides.cancellation_reason_id"
    },
    {
        "table": "uber.bronze.map_types as map_types",
        "select" : "map_types.*",
        "where" : "",
        "on": "stg_rides.vehicle_make_id = map_types.vehicle_make_id"
    }
    
]

In [0]:
from jinja2 import Template
jinja_str="""

select
{% for config in jinja_config %}
  {{config.select}}
     {% if not loop.last %}
     ,
     {% endif %}
{% endfor %}
from
    {% for config in jinja_config %}
      {% if loop.first %}
        {{config.table}}
      {% else %}
        left join {{config.table}} on {{config.on}}
      {% endif %}
    {% endfor %}

    {% for config in jinja_config %}
      {% if loop.first %}
         {% if config.where != "" %}
           where
         {% endif %}
      {% endif %}
      {% if config.where != "" %}
        {{config.where}}
      {% endif %}
    {% endfor %}

"""

template = Template(jinja_str)
rendered_template = template.render(jinja_config=jinja_config)
print(rendered_template)



In [0]:
spark.sql(rendered_template).display()

In [0]:
%sql
select * from uber.bronze.silver_obt